# EchoFace 02: Wav2Lip on your own photo and audio

Turn a still photo plus a speech clip into a lip-synced video.

**What you need**
- A clear, front-facing photo of a face. Use your own photo, or one of someone who has agreed to it.
- A short speech recording (`.wav` or `.mp3`). Start with 10 to 30 seconds.

**Set up**
- Runtime, Change runtime type, **T4 GPU**, then run the cells from top to bottom.
- Unlike notebook 01, this one does **not** downgrade `numpy` or `librosa`. It patches four lines in Wav2Lip instead, so it works on Colab's current Python.

**Licence:** Wav2Lip is for personal, research and non-commercial use only.

## 1. Check the environment

In [ ]:
import sys, torch
print("Python:", sys.version.split()[0])
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("No GPU found. Use Runtime > Change runtime type > T4 GPU, then rerun from the top.")
!ffmpeg -version | head -1

## 2. Get Wav2Lip (pinned to a known version)

In [ ]:
%cd /content
!rm -rf Wav2Lip
!git clone -q https://github.com/Rudrabha/Wav2Lip.git
!cd Wav2Lip && git checkout -q bac9a81e63ecc153202353372e5724b83d9e6322 && git log -1 --format='Using Wav2Lip commit %h from %cd'

## 3. Patch Wav2Lip for current libraries

Wav2Lip was written in 2020. Four lines break on today's `librosa`, `numpy` and `PyTorch`:

| File | Problem | Fix |
|---|---|---|
| `audio.py` | `librosa.filters.mel` now needs keyword arguments (the `TypeError` from before) | pass `sr=` and `n_fft=` by name |
| `inference.py` | PyTorch 2.6+ refuses to load the checkpoint by default | add `weights_only=False` |
| `face_detection/.../sfd_detector.py` | same | same |
| `face_detection/utils.py` | `np.int` was removed from numpy | use `int` |

Only load checkpoints you trust, because `weights_only=False` allows arbitrary code in a checkpoint file.

In [ ]:
import pathlib, re

W2L = pathlib.Path("/content/Wav2Lip")

def patch(rel_path, pattern, replacement, already_patched):
    """Apply one regex fix. Safe to run twice. Stops loudly if the code is not what we expect."""
    f = W2L / rel_path
    text = f.read_text()
    new_text, n = re.subn(pattern, replacement, text)
    if n:
        f.write_text(new_text)
        print(f"patched  {rel_path}  ({n} change{'s' if n > 1 else ''})")
    elif already_patched in text:
        print(f"already  {rel_path}")
    else:
        raise RuntimeError(f"Could not patch {rel_path}: the Wav2Lip code is not what this notebook expects.")

# 1) librosa 0.10+ only accepts keyword arguments for the mel filterbank (this was the TypeError).
patch("audio.py",
      r"librosa\.filters\.mel\(hp\.sample_rate, hp\.n_fft,",
      "librosa.filters.mel(sr=hp.sample_rate, n_fft=hp.n_fft,",
      "mel(sr=hp.sample_rate")

# 2) PyTorch 2.6+ refuses to unpickle checkpoints unless weights_only=False is stated.
def add_weights_only(rel_path):
    f = W2L / rel_path
    text = f.read_text()
    def fix(m):
        return m.group(0) if "weights_only" in m.group(0) else f"torch.load({m.group(1)}, weights_only=False)"
    new_text, n = re.subn(r"torch\.load\(([^()]*)\)", fix, text)
    if new_text != text:
        f.write_text(new_text)
        print(f"patched  {rel_path}  (torch.load)")
    elif "weights_only" in text:
        print(f"already  {rel_path}")
    else:
        raise RuntimeError(f"No torch.load found in {rel_path}")

add_weights_only("inference.py")
add_weights_only("face_detection/detection/sfd/sfd_detector.py")

# 3) numpy 1.24+ removed np.int.
patch("face_detection/utils.py", r"dtype=np\.int\)", "dtype=int)", "dtype=int)")
print("Wav2Lip is ready for current library versions.")

## 4. Download the model weights

In [ ]:
import os, subprocess
os.makedirs("/content/Wav2Lip/checkpoints", exist_ok=True)
os.makedirs("/content/Wav2Lip/face_detection/detection/sfd", exist_ok=True)

def fetch(url, dest, min_mb):
    if os.path.exists(dest) and os.path.getsize(dest) > min_mb * 1e6:
        print("already have", os.path.basename(dest)); return
    subprocess.run(["wget", "-q", "--show-progress", "-U", "Mozilla/5.0", "-O", dest, url], check=False)
    size = os.path.getsize(dest) / 1e6 if os.path.exists(dest) else 0
    if size < min_mb:
        raise RuntimeError(f"Download of {os.path.basename(dest)} failed or is incomplete ({size:.1f} MB). Open the Wav2Lip README, download it manually and upload it to {dest}.")
    print(f"downloaded {os.path.basename(dest)}: {size:.0f} MB")

# Lip-sync generator with the GAN refinement (community mirror of the official weights, about 436 MB).
fetch("https://github.com/justinjohn0306/Wav2Lip/releases/download/models/wav2lip_gan.pth",
      "/content/Wav2Lip/checkpoints/wav2lip_gan.pth", 400)
# Face detector (S3FD).
fetch("https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth",
      "/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth", 50)

## 5. Upload your photo and audio

Photo tips: one face, looking at the camera, mouth visible, even lighting, at least 256 px tall.
By uploading a photo you confirm you have the right to use that person's face and voice.

In [ ]:
from google.colab import files
import os, shutil
os.makedirs("/content/inputs", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

print("Upload your PHOTO (.jpg or .png):")
up = files.upload()
name = list(up.keys())[0]
FACE_RAW = f"/content/inputs/raw_{len(name)}_{os.path.basename(name)}"
shutil.move(name, FACE_RAW)

print("Upload your AUDIO (.wav, .mp3 or .m4a):")
up = files.upload()
name = list(up.keys())[0]
AUDIO_RAW = f"/content/inputs/raw_{len(name)}_{os.path.basename(name)}"
shutil.move(name, AUDIO_RAW)
print("Received:", os.path.basename(FACE_RAW), "and", os.path.basename(AUDIO_RAW))

### Prepare the inputs

Wav2Lip runs shell commands with the file names unquoted, so names with spaces or brackets, like `Recording (1).m4a`, make it crash. This step copies your files to plain names (`face.jpg`, `audio.wav`) and converts the audio to 16 kHz mono. Only the first 30 seconds of audio are used.

In [ ]:
import cv2, subprocess
MAX_SECONDS = 30
FACE_PATH = "/content/inputs/face.jpg"
AUDIO_PATH = "/content/inputs/audio.wav"

img = cv2.imread(FACE_RAW)
if img is None:
    raise RuntimeError("Could not read the photo. Use a .jpg or .png file (iPhone HEIC photos need converting first).")
cv2.imwrite(FACE_PATH, img)

r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", AUDIO_RAW, "-t", str(MAX_SECONDS),
                    "-ac", "1", "-ar", "16000", AUDIO_PATH], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("Could not read the audio file:\n" + r.stderr)
print("Photo ready:", FACE_PATH, f"({img.shape[1]} x {img.shape[0]})")
print("Audio ready:", AUDIO_PATH)

In [ ]:
import cv2, matplotlib.pyplot as plt
import IPython.display as ipd

img = cv2.imread(FACE_PATH)
if img is None:
    raise RuntimeError("Could not read the photo. Use a .jpg or .png file.")
print("Photo size:", img.shape[1], "x", img.shape[0])
plt.figure(figsize=(4, 4)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
ipd.display(ipd.Audio(AUDIO_PATH))

## 6. Generate the video

In [ ]:
OUT_PATH = "/content/outputs/echoface_result.mp4"
!cd /content/Wav2Lip && python inference.py \
    --checkpoint_path checkpoints/wav2lip_gan.pth \
    --face "{FACE_PATH}" \
    --audio "{AUDIO_PATH}" \
    --outfile "{OUT_PATH}" \
    --pads 0 10 0 0 \
    --face_det_batch_size 4

If you see `Face not detected`, use a clearer, closer, front-facing photo. If you see `out of memory`, add `--resize_factor 2` to the command above.

## 7. Watch it and save it

In [ ]:
import os
from base64 import b64encode
from IPython.display import HTML
assert os.path.exists(OUT_PATH), "No output file. Read the error in the previous cell."
data_url = "data:video/mp4;base64," + b64encode(open(OUT_PATH, "rb").read()).decode()
HTML(f'<video width="420" controls autoplay loop><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
from google.colab import files
files.download(OUT_PATH)

## 8. What I noticed

Write your own notes after watching the result:
- Does the mouth open and close at the right moments?
- How sharp is the mouth compared with the rest of the face?
- Is there a visible box or seam around the mouth?
- What changed between a photo and a video of the same person (notebook 01)?

Ideas to try next: a different photo angle, noisy versus clean audio, and the non-GAN checkpoint (`wav2lip.pth`) to compare sync against sharpness.